# Neural Networks: Hyperparameter Selection with a Validation Set

Part 1 classifies the two moons. We choose the learning rate and the architecture with a validation set and
report the error on a test set.
Part 2 fits the noisy function $\sin(3x) + \cos(5x)$ with a fixed learning rate and architecture.

The three sets have different roles:

| set | used to |
|---|---|
| training | fit the weights (gradient descent) |
| validation | choose hyperparameters: learning rate, architecture |
| test | report the final error, once, at the end |

Cells marked `# TODO` must be completed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import make_moons

rng = np.random.default_rng(0)
CMAP = "bwr"
to_t = lambda a: torch.tensor(a, dtype=torch.float32)

# Part 1. Classification of the two moons

## 1.1 The data

600 points split 60% / 20% / 20% into training, validation and test.
No straight line separates the two classes.

In [ ]:
X, y = make_moons(600, noise=0.2, random_state=0)
idx = rng.permutation(len(X))
# TODO: split idx into 360 training, 120 validation and 120 test indices
i_tr, i_va, i_te = ...

X_tr, X_va, X_te = to_t(X[i_tr]), to_t(X[i_va]), to_t(X[i_te])
y_tr, y_va, y_te = (to_t(y[i])[:, None] for i in (i_tr, i_va, i_te))

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, (Xs, ys, name) in zip(ax, [(X_tr, y_tr, "train"), (X_va, y_va, "validation"), (X_te, y_te, "test")]):
    a.scatter(Xs[:, 0], Xs[:, 1], c=ys.ravel(), cmap=CMAP, edgecolor="k", s=15)
    a.set(title=f"{name} ({len(Xs)} points)", aspect="equal")
plt.show()

## 1.2 Model and training loop

`make_mlp([2, 8, 1])` builds a network with layer sizes 2 -> 8 -> 1 and a $\tanh$ after every hidden layer.
The output is the logit; `nn.BCEWithLogitsLoss` applies the sigmoid and the cross-entropy.

Every training loop has the same five lines:

| line | meaning |
|---|---|
| `out = model(X)` | forward pass |
| `loss = loss_fn(out, y)` | how wrong are we |
| `opt.zero_grad()` | reset accumulated gradients |
| `loss.backward()` | compute $\nabla L$ for every weight |
| `opt.step()` | $w \leftarrow w - \eta \nabla L$ |

Training is full batch: one epoch is one gradient step on the whole training set.
After every epoch we record the training and validation loss.

In [ ]:
def make_mlp(sizes, seed=0):
    torch.manual_seed(seed)
    layers = []
    # TODO: architecture of the nn
    ...


def train(model, loss_fn, X_tr, y_tr, X_va, y_va, lr, epochs):
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    hist = {"train": [], "val": []}
    for epoch in range(epochs):
        # TODO: forward pass, loss, zero_grad, backward, step
        ...
        with torch.no_grad():
            hist["train"].append(loss_fn(model(X_tr), y_tr).item())
            hist["val"].append(loss_fn(model(X_va), y_va).item())
    return hist


def error_rate(model, X, y):
    # TODO: fraction of points where (logit > 0) differs from y
    ...

Plotting helpers.

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
grid = to_t(np.c_[g1.ravel(), g2.ravel()])


def plot_boundary(ax, model, X, y, title=""):
    with torch.no_grad():
        prob = torch.sigmoid(model(grid)).numpy().reshape(g1.shape)
    ax.contourf(g1, g2, prob, levels=np.linspace(0, 1, 11), cmap=CMAP, alpha=0.4)
    ax.contour(g1, g2, prob, levels=[0.5], colors="k")
    ax.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap=CMAP, edgecolor="k", s=15)
    ax.set(xlim=(-2, 3), ylim=(-1.5, 2), aspect="equal", title=title)


def plot_curves(ax, hist, title="", ylabel="loss"):
    ax.plot(hist["train"], label="train")
    ax.plot(hist["val"], label="validation")
    ax.set(yscale="log", xlabel="epoch", ylabel=ylabel, title=title)
    ax.legend()

## 1.3 One network

Before searching, train a single network with one hidden layer of 8 neurons and `lr = 1.0`.

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
# TODO: build a [2, 8, 1] network and train it with lr = 1.0 for 1000 epochs
model = ...
hist = ...

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
plot_boundary(ax[0], model, X_va, y_va, f"validation error = {error_rate(model, X_va, y_va):.3f}")
plot_curves(ax[1], hist, title="cross-entropy vs epoch")
plt.show()

## 1.4 Choosing the learning rate and the architecture

We train every combination of learning rate and architecture on the **training** set
and compare them on the **validation** set. The test set is not used here.

If a run diverges its loss becomes `nan`; we count it as infinitely bad.

In [ ]:
lrs = [0.01, 0.1, 1.0, 10.0]
archs = {"2-2-1": [2, 2, 1], "2-8-1": [2, 8, 1], "2-8-8-1": [2, 8, 8, 1]}
EPOCHS = 1000

results = {}  # (arch, lr) -> (model, hist)
# TODO: for every architecture and every learning rate, build a network, train it
# and store (model, hist) in results[(name, lr)]


val_loss = np.full((len(archs), len(lrs)), np.inf)
for i, name in enumerate(archs):
    for j, lr in enumerate(lrs):
        model, hist = results[(name, lr)]
        if np.isfinite(hist["val"][-1]):
            val_loss[i, j] = hist["val"][-1]
        print(f"{name:8s} lr = {lr:<5} train loss = {hist['train'][-1]:.3f}  val loss = {hist['val'][-1]:.3f}"
              f"  val error = {error_rate(model, X_va, y_va):.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
im = ax[0].imshow(np.log10(np.where(np.isfinite(val_loss), val_loss, np.nan)), cmap="viridis_r")
ax[0].set(xticks=range(len(lrs)), xticklabels=lrs, yticks=range(len(archs)), yticklabels=list(archs),
          xlabel="learning rate", ylabel="architecture", title="log10 validation loss (white = diverged)")
fig.colorbar(im, ax=ax[0])

for lr in lrs:
    ax[1].plot(results[("2-8-8-1", lr)][1]["val"], label=f"lr = {lr}")
ax[1].set(yscale="log", xlabel="epoch", ylabel="cross-entropy", title="validation loss, 2-8-8-1")
ax[1].legend()
plt.show()

- Which learning rate is too small? How can you tell from the curve?
- What happens with `lr = 10`?
- Does the deeper network always win?

## 1.5 Final evaluation on the test set

Select the combination with the lowest validation loss and evaluate it on the test set **once**.

In [ ]:
# TODO: find the (architecture, learning rate) with the lowest validation loss
# hint: np.argmin and np.unravel_index
i, j = ...
best_arch, best_lr = list(archs)[i], lrs[j]
best_model, best_hist = results[(best_arch, best_lr)]

# TODO: test error of the selected model
test_err = ...
print(f"selected: {best_arch}, lr = {best_lr}")
print(f"validation error = {error_rate(best_model, X_va, y_va):.3f}   test error = {test_err:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
plot_boundary(ax[0], best_model, X_te, y_te, f"{best_arch}, lr = {best_lr}, test error = {test_err:.3f}")
plot_curves(ax[1], best_hist, title="selected model")
plt.show()

- Why should we not choose the model by its test error?
- Ususally the validation error of the selected model is slightly optimistic. Why?

# Part 2. Regression: fitting $\sin(3x) + \cos(5x)$

## 2.1 The data

$$y = f(x) + \varepsilon, \qquad f(x) = \sin(3x) + \cos(5x), \qquad \varepsilon \sim \mathcal{N}(0, \sigma^2), \quad \sigma = 0.3$$

300 points with $x \in [-1.5, 1.5]$, split 180 / 60 / 60. The loss is the mean squared error.
Even the true $f$ has MSE $\approx \sigma^2 = 0.09$ on new data: this **noise level** is the best
validation or test MSE we can expect.

In [ ]:
SIGMA = 0.3


def f_true(x):
    return np.sin(3 * x) + np.cos(5 * x)


x_all = rng.uniform(-1.5, 1.5, 300)
y_all = f_true(x_all) + SIGMA * rng.normal(size=300)
s_tr, s_va, s_te = slice(0, 180), slice(180, 240), slice(240, 300)
x_tr, x_va, x_te = (to_t(x_all[s])[:, None] for s in (s_tr, s_va, s_te))
t_tr, t_va, t_te = (to_t(y_all[s])[:, None] for s in (s_tr, s_va, s_te))

xs = np.linspace(-1.5, 1.5, 400)
plt.figure(figsize=(9, 4))
plt.plot(xs, f_true(xs), "k", lw=2, label="f(x)")
for xx, yy, name in [(x_tr, t_tr, "train"), (x_va, t_va, "validation"), (x_te, t_te, "test")]:
    plt.scatter(xx, yy, s=12, label=name)
plt.legend()
plt.show()

## 2.2 Training with a fixed learning rate

Network 1 -> 16 -> 16 -> 16 -> 1, plain gradient descent with `lr = 0.1`, 3000 epochs.
The same `make_mlp` and `train` work here: only the loss changes, to `nn.MSELoss()`.

In [ ]:
mse = nn.MSELoss()
# TODO: build a [1, 16, 16, 16, 1] network and train it with lr = 0.1 for 3000 epochs
reg = ...
hist_r = ...

with torch.no_grad():
    y_plot = reg(to_t(xs)[:, None]).numpy()
    # TODO: test MSE
    test_mse = ...

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(xs, y_plot, "r", lw=2, label="network")
ax[0].plot(xs, f_true(xs), "k--", lw=1, label="f(x)")
ax[0].scatter(x_tr, t_tr, s=10, c="gray", alpha=0.6, label="train")
ax[0].set(xlabel="x", ylabel="y", title=f"test MSE = {test_mse:.3f}")
ax[0].legend()
plot_curves(ax[1], hist_r, title="MSE vs epoch", ylabel="MSE")
ax[1].axhline(SIGMA ** 2, color="k", ls="--", lw=1, label=r"noise level $\sigma^2$")
ax[1].legend()
plt.show()
print(f"train MSE = {hist_r['train'][-1]:.3f}  val MSE = {hist_r['val'][-1]:.3f}  test MSE = {test_mse:.3f}"
      f"  noise level = {SIGMA ** 2:.3f}")

- Is the test MSE close to the noise level?
- Is the training MSE below the noise level? What would it mean if it were much lower?
- Exercise: repeat 1.4 for this problem (learning rates `[0.01, 0.1, 0.3]`, a few architectures) and select by validation MSE.

# Summary

- Training set: fit the weights. Validation set: choose learning rate and architecture. Test set: report the error once.
- A learning rate that is too small leaves the network unfinished; one that is too large diverges.
- The same training loop works for classification and regression; only the loss changes.